In [ ]:


import os, zipfile, random, shutil, math
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.applications import DenseNet121, ResNet50, EfficientNetB0
from tensorflow.keras.applications.densenet    import preprocess_input as pre_dn
from tensorflow.keras.applications.resnet50    import preprocess_input as pre_rn
from tensorflow.keras.applications.efficientnet import preprocess_input as pre_en
from tensorflow.keras import layers, models
from sklearn.model_selection      import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing        import StandardScaler
from sklearn.decomposition        import PCA
from sklearn.svm                  import SVC
from sklearn.linear_model         import LogisticRegression
from sklearn.ensemble             import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neighbors            import KNeighborsClassifier
from sklearn.calibration          import CalibratedClassifierCV
from sklearn.metrics              import classification_report, confusion_matrix, roc_auc_score

SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)
IMG_SIZE   = 224
BATCH_SIZE = 32

FGNET_ZIP = "/content/FGNET.zip"
MAL_ZIP   = next((c for c in ("/content/content.zip", "/content/clean_dataset.zip")
                  if os.path.exists(c)), None)
if MAL_ZIP is None:
    raise FileNotFoundError("Upload /content/content.zip or /content/clean_dataset.zip")

with zipfile.ZipFile(FGNET_ZIP) as z: z.extractall("/content/FGNET")
with zipfile.ZipFile(MAL_ZIP)   as z: z.extractall("/content/malnutrition_raw")
print(f"Extracted — {os.path.basename(MAL_ZIP)}")


classes = ["Normal Faces", "abnormal Faces"]

def find_dataset_root(start):
    for dirpath, dirnames, _ in os.walk(start):
        if "__MACOSX" in dirpath: continue
        dirnames[:] = [d for d in dirnames if d != "__MACOSX"]
        for sub in dirnames:
            cand = os.path.join(dirpath, sub)
            if all(os.path.isdir(os.path.join(cand, c)) for c in classes):
                return dirpath if sub.lower() in ("train","validation","val","test") else cand
    return None

src_root = find_dataset_root("/content/malnutrition_raw")
assert src_root, "Cannot find class folders."
print(f"Dataset root: {src_root}")

def _is_img(f):
    return not f.startswith(".") and f.lower().endswith(('.jpg','.jpeg','.png'))

all_paths, all_labels = [], []
for idx, cls in enumerate(classes):
    found = False
    for split in ("train","validation","val","test"):
        d = os.path.join(src_root, split, cls)
        if os.path.isdir(d):
            found = True
            for f in os.listdir(d):
                if _is_img(f): all_paths.append(os.path.join(d,f)); all_labels.append(idx)
    if not found:
        d = os.path.join(src_root, cls)
        if os.path.isdir(d):
            for f in os.listdir(d):
                if _is_img(f): all_paths.append(os.path.join(d,f)); all_labels.append(idx)

print(f"Total images: {len(all_paths)}")
train_p, val_p, train_y, val_y = train_test_split(
    all_paths, all_labels, test_size=0.20, stratify=all_labels, random_state=SEED)
y_train = np.array(train_y)
y_val   = np.array(val_y)
print(f"Train {len(train_p)} | Val {len(val_p)}")
print(f"  Train — Normal {(y_train==0).sum()} | Abnormal {(y_train==1).sum()}")
print(f"  Val   — Normal {(y_val==0).sum()}   | Abnormal {(y_val==1).sum()}")

def load_imgs_raw(paths):
    """Load images as float32 [N,224,224,3] in [0,255]."""
    out = []
    for p in paths:
        raw = tf.io.read_file(p)
        img = tf.image.decode_jpeg(raw, channels=3)
        img = tf.image.resize(img, [IMG_SIZE, IMG_SIZE])
        out.append(img.numpy())
    return np.array(out, dtype=np.float32)

def augment_numpy(imgs, seed=None):
    """
    Light augmentation for feature-level TTA.
    imgs: float32 [N,224,224,3] in [0,255].
    Returns augmented copy (same shape), still in [0,255].
    """
    if seed is not None:
        np.random.seed(seed)
        random.seed(seed)
    out = imgs.copy()
    for i in range(len(out)):
        img = out[i]

        if random.random() > 0.5:
            img = img[:, ::-1, :]

        img = img + random.uniform(-0.12, 0.12) * 255.
        img = np.clip(img, 0., 255.)

        mean = img.mean()
        img  = (img - mean) * random.uniform(0.88, 1.12) + mean
        img  = np.clip(img, 0., 255.)

        gray = img.mean(axis=2, keepdims=True)
        img  = gray + (img - gray) * random.uniform(0.80, 1.20)
        img  = np.clip(img, 0., 255.)

        frac = random.uniform(0.90, 1.0)
        c    = int(IMG_SIZE * frac)
        oy   = random.randint(0, IMG_SIZE - c)
        ox   = random.randint(0, IMG_SIZE - c)
        crop = img[oy:oy+c, ox:ox+c, :]
        img  = tf.image.resize(crop[np.newaxis], [IMG_SIZE, IMG_SIZE])[0].numpy()
        out[i] = np.clip(img, 0., 255.)
    return out

print("\nBuilding frozen feature extractors…")

def make_extractor(backbone_cls, name):
    base = backbone_cls(weights='imagenet', include_top=False,
                        input_shape=(IMG_SIZE, IMG_SIZE, 3))
    for layer in base.layers:
        layer.trainable = False
    out = layers.GlobalAveragePooling2D()(base.output)
    mdl = models.Model(base.input, out, name=name)
    print(f"  {name}: {int(out.shape[-1])}-d")
    return mdl

dn_ext  = make_extractor(DenseNet121,   "densenet121")
rn_ext  = make_extractor(ResNet50,      "resnet50")
en_ext  = make_extractor(EfficientNetB0,"efficientnetb0")

def extract_once(imgs_raw, extractor, preprocess_fn):
    """Extract GAP features from a single pass (no augmentation)."""
    imgs = preprocess_fn(imgs_raw.copy())
    ds   = tf.data.Dataset.from_tensor_slices(imgs).batch(BATCH_SIZE)
    return extractor.predict(ds, verbose=0)

N_TTA = 15

print(f"\nExtracting features with {N_TTA}-view TTA…")

def extract_tta_features(paths, extractor, preprocess_fn, n_tta=N_TTA):
    """
    Returns averaged features [N_images, feat_dim].
    Pass 0 = clean (no augmentation).
    Passes 1..n_tta-1 = randomly augmented.
    """
    imgs_raw = load_imgs_raw(paths)

    feats = extract_once(imgs_raw, extractor, preprocess_fn).astype(np.float64)

    for aug_idx in range(1, n_tta):
        aug_imgs = augment_numpy(imgs_raw, seed=aug_idx * 1000 + len(paths))
        feats   += extract_once(aug_imgs, extractor, preprocess_fn).astype(np.float64)
    return (feats / n_tta).astype(np.float32)


print("  DenseNet121 TTA…")
dn_train_tta = extract_tta_features(train_p, dn_ext, pre_dn)
dn_val_tta   = extract_tta_features(val_p,   dn_ext, pre_dn)

print("  ResNet50 TTA…")
rn_train_tta = extract_tta_features(train_p, rn_ext, pre_rn)
rn_val_tta   = extract_tta_features(val_p,   rn_ext, pre_rn)

print("  EfficientNetB0 TTA…")
en_train_tta = extract_tta_features(train_p, en_ext, pre_en)
en_val_tta   = extract_tta_features(val_p,   en_ext, pre_en)

X_train_all = np.concatenate([dn_train_tta, rn_train_tta, en_train_tta], axis=1)
X_val_all   = np.concatenate([dn_val_tta,   rn_val_tta,   en_val_tta],   axis=1)
print(f"  Concatenated TTA feature dim: {X_train_all.shape[1]}")

print("\n── PCA sweep on concatenated TTA features ──")

sc_all = StandardScaler()
Xtr_scaled = sc_all.fit_transform(X_train_all)
Xvl_scaled = sc_all.transform(X_val_all)

best_pca_n, best_pca_cv = 64, 0.0
for n_comp in [32, 64, 96, 128, 192, 256, 320]:
    pca_tmp = PCA(n_components=n_comp, random_state=SEED)
    Xtr_pca = pca_tmp.fit_transform(Xtr_scaled)
    svm_tmp = SVC(kernel='rbf', C=10.0, gamma='scale',
                  class_weight='balanced', random_state=SEED)
    cv = cross_val_score(svm_tmp, Xtr_pca, y_train, cv=5,
                         scoring='balanced_accuracy').mean()
    var = pca_tmp.explained_variance_ratio_.sum()
    print(f"  PCA n={n_comp:3d} | var_explained={var:.3f} | 5-fold bal_acc={cv:.4f}")
    if cv > best_pca_cv:
        best_pca_cv, best_pca_n = cv, n_comp

print(f"\n  Best PCA n={best_pca_n} (CV bal_acc={best_pca_cv:.4f})")
pca_best = PCA(n_components=best_pca_n, random_state=SEED)
Xtr_pca_best = pca_best.fit_transform(Xtr_scaled)
Xvl_pca_best = pca_best.transform(Xvl_scaled)

print("\n── Multiple classifiers ──")

def cv_tune_svm_C(X, y, kernel='rbf'):
    best_c, best_cv = 1.0, 0.0
    for C in [0.01, 0.1, 0.5, 1.0, 5.0, 10.0, 50.0, 100.0]:
        svm = SVC(kernel=kernel, C=C, gamma='scale' if kernel=='rbf' else 'auto',
                  class_weight='balanced', random_state=SEED)
        cv = cross_val_score(svm, X, y, cv=5, scoring='balanced_accuracy').mean()
        if cv > best_cv: best_cv, best_c = cv, C
    return best_c, best_cv

def make_all_classifiers(X_tr, y_tr, tag):
    """Train a suite of classifiers, return list of (name, calibrated_clf, cv_score)."""
    results = []
    sc = StandardScaler()
    Xtr = sc.fit_transform(X_tr)

    C_rbf, cv_rbf = cv_tune_svm_C(Xtr, y_tr, kernel='rbf')
    svm_rbf = SVC(kernel='rbf', C=C_rbf, gamma='scale',
                  class_weight='balanced', probability=True, random_state=SEED)
    cal_rbf = CalibratedClassifierCV(svm_rbf, cv=5, method='sigmoid')
    cal_rbf.fit(Xtr, y_tr)
    results.append((f"RBF-SVM[{tag}]", cal_rbf, sc, cv_rbf))


    C_lin, cv_lin = cv_tune_svm_C(Xtr, y_tr, kernel='linear')
    svm_lin = SVC(kernel='linear', C=C_lin,
                  class_weight='balanced', probability=True, random_state=SEED)
    cal_lin = CalibratedClassifierCV(svm_lin, cv=5, method='sigmoid')
    cal_lin.fit(Xtr, y_tr)
    results.append((f"Linear-SVM[{tag}]", cal_lin, sc, cv_lin))

    lr = LogisticRegression(C=1.0, max_iter=1000, class_weight='balanced',
                            random_state=SEED, solver='lbfgs')
    cv_lr = cross_val_score(lr, Xtr, y_tr, cv=5, scoring='balanced_accuracy').mean()
    lr.fit(Xtr, y_tr)
    results.append((f"LogReg[{tag}]", lr, sc, cv_lr))

    rf = RandomForestClassifier(n_estimators=500, max_features='sqrt',
                                class_weight='balanced', random_state=SEED, n_jobs=-1)
    cv_rf = cross_val_score(rf, Xtr, y_tr, cv=5, scoring='balanced_accuracy').mean()
    rf.fit(Xtr, y_tr)
    results.append((f"RF[{tag}]", rf, sc, cv_rf))

    gb = GradientBoostingClassifier(n_estimators=200, max_depth=3,
                                    learning_rate=0.05, random_state=SEED)
    cv_gb = cross_val_score(gb, Xtr, y_tr, cv=5, scoring='balanced_accuracy').mean()
    gb.fit(Xtr, y_tr)
    results.append((f"GBM[{tag}]", gb, sc, cv_gb))

    return results

all_results = []
for feat_train, feat_val, tag in [
    (dn_train_tta, dn_val_tta,   "DenseNet-TTA"),
    (rn_train_tta, rn_val_tta,   "ResNet-TTA"),
    (en_train_tta, en_val_tta,   "EfficientNet-TTA"),
]:
    print(f"\n  Training on {tag} features…")
    res = make_all_classifiers(feat_train, y_train, tag)
    for name, clf, sc, cv in res:
        Xvl = sc.transform(feat_val)
        probs = clf.predict_proba(Xvl)[:, 1]
        val_acc = ((probs >= 0.5).astype(int) == y_val).mean()
        print(f"    {name:<35s} CV={cv:.4f}  val_acc={val_acc*100:.2f}%")
        all_results.append((name, clf, sc, feat_val, probs, cv, val_acc))

print(f"\n  Training on PCA-{best_pca_n} concat-TTA features…")
for clf_name, clf_obj, sc_pca, cv_score in [
    ("RBF-SVM", SVC(kernel='rbf', gamma='scale', class_weight='balanced',
                    probability=True, random_state=SEED), StandardScaler(), 0.),
    ("Linear-SVM", SVC(kernel='linear', class_weight='balanced',
                       probability=True, random_state=SEED), StandardScaler(), 0.),
    ("LogReg", LogisticRegression(C=1.0, max_iter=1000, class_weight='balanced',
                                  random_state=SEED), StandardScaler(), 0.),
    ("GBM", GradientBoostingClassifier(n_estimators=200, max_depth=3,
                                       learning_rate=0.05, random_state=SEED),
     StandardScaler(), 0.),
]:
    if "SVM" in clf_name:
        k = 'rbf' if "RBF" in clf_name else 'linear'
        C_opt, cv_score = cv_tune_svm_C(Xtr_pca_best, y_train, kernel=k)
        clf_obj.C = C_opt
    else:
        cv_score = cross_val_score(clf_obj, Xtr_pca_best, y_train,
                                   cv=5, scoring='balanced_accuracy').mean()
    clf_obj.fit(Xtr_pca_best, y_train)
    probs_v = clf_obj.predict_proba(Xvl_pca_best)[:, 1]
    val_acc  = ((probs_v >= 0.5).astype(int) == y_val).mean()
    tag_name = f"{clf_name}[PCA-{best_pca_n}-concat-TTA]"
    print(f"    {tag_name:<45s} CV={cv_score:.4f}  val_acc={val_acc*100:.2f}%")
    all_results.append((tag_name, clf_obj, None, Xvl_pca_best, probs_v, cv_score, val_acc))


print("\n── Rank-based ensemble ──")
all_results.sort(key=lambda x: x[5], reverse=True)

print("\nAll classifiers ranked by 5-fold CV balanced_accuracy:")
print(f"  {'Name':<45s} {'CV':>7}  {'val_acc':>8}")
print(f"  {'-'*63}")
for name, _, _, _, probs, cv, val_acc in all_results:
    print(f"  {name:<45s} {cv:7.4f}  {val_acc*100:7.2f}%")

best_cv = all_results[0][5]
keep    = [(name, probs, cv, val_acc)
           for name, _, _, _, probs, cv, val_acc in all_results
           if cv >= best_cv - 0.03]
print(f"\nKeeping {len(keep)} classifiers (CV >= {best_cv-0.03:.4f}):")
for name, probs, cv, val_acc in keep:
    print(f"  {name:<45s} CV={cv:.4f}  val_acc={val_acc*100:.2f}%")


def threshold_sweep(probs, y):
    best_thr, best_bal = 0.5, 0.0
    for t in np.arange(0.20, 0.81, 0.01):
        p   = (probs >= t).astype(int)
        tp  = int(((p==1)&(y==1)).sum()); tn = int(((p==0)&(y==0)).sum())
        fn  = int(((p==0)&(y==1)).sum()); fp = int(((p==1)&(y==0)).sum())
        bal = 0.5*(tp/max(tp+fn,1) + tn/max(tn+fp,1))
        if bal > best_bal: best_bal, best_thr = bal, t
    return best_thr, best_bal

def report(probs, tag, thr=None):
    if thr is None: thr, _ = threshold_sweep(probs, y_val)
    pred = (probs >= thr).astype(int)
    acc  = (pred == y_val).mean()
    auc  = roc_auc_score(y_val, probs)
    sep  = probs[y_val==1].mean() - probs[y_val==0].mean()
    print(f"\n{'═'*62}")
    print(f"  {tag}")
    print(f"  threshold={thr:.2f} | AUC={auc:.4f} | sep={sep:+.3f}")
    print(f"{'═'*62}")
    print(classification_report(y_val, pred,
          target_names=['Normal','Abnormal'], digits=3, zero_division=0))
    print(f"Confusion matrix:\n{confusion_matrix(y_val, pred)}")
    print(f"Accuracy: {acc*100:.2f}%")
    return acc, thr

print("\n── Individual top classifier reports ──")
for name, probs, cv, val_acc in keep[:5]:
    report(probs, name)

if len(keep) >= 2:
    weights  = np.array([cv for _, _, cv, _ in keep])
    weights  = weights / weights.sum()
    prob_mat = np.column_stack([probs for _, probs, _, _ in keep])
    ens_probs_weighted = (prob_mat * weights).sum(axis=1)
    acc_wt, thr_wt = report(ens_probs_weighted,
                             f"Weighted ensemble ({len(keep)} classifiers)")

ens_probs_avg = prob_mat.mean(axis=1)
acc_avg, thr_avg = report(ens_probs_avg,
                           f"Simple-average ensemble ({len(keep)} classifiers)")

best_ens_probs = ens_probs_weighted if acc_wt >= acc_avg else ens_probs_avg
best_ens_acc   = max(acc_wt, acc_avg)
best_ens_thr   = thr_wt if acc_wt >= acc_avg else thr_avg

best_single     = all_results[0]
single_probs    = best_single[4]
acc_single, thr_single = report(single_probs, f"BEST SINGLE: {best_single[0]}")

print(f"  FINAL SUMMARY")
print(f"  Best single classifier : {all_results[0][0]}")
print(f"                           {acc_single*100:.2f}%  (thr={thr_single:.2f})")
print(f"  Weighted ensemble      : {acc_wt*100:.2f}%  (thr={thr_wt:.2f})")
print(f"  Simple-avg ensemble    : {acc_avg*100:.2f}%  (thr={thr_avg:.2f})")
overall_best = max(acc_single, acc_wt, acc_avg)
print(f"  ★ BEST OVERALL         : {overall_best*100:.2f}%")

Extracted — clean_dataset.zip
Dataset root: /content/malnutrition_raw/content/clean_dataset/dataFinal
Total images: 446
Train 356 | Val 90
  Train — Normal 222 | Abnormal 134
  Val   — Normal 56   | Abnormal 34

Building frozen feature extractors…
  densenet121: 1024-d
  resnet50: 2048-d
  efficientnetb0: 1280-d

Extracting features with 15-view TTA…
  DenseNet121 TTA…
  ResNet50 TTA…
  EfficientNetB0 TTA…
  Concatenated TTA feature dim: 4352

── PCA sweep on concatenated TTA features ──
  PCA n= 32 | var_explained=0.571 | 5-fold bal_acc=0.6249
  PCA n= 64 | var_explained=0.713 | 5-fold bal_acc=0.6316
  PCA n= 96 | var_explained=0.794 | 5-fold bal_acc=0.5974
  PCA n=128 | var_explained=0.849 | 5-fold bal_acc=0.6318
  PCA n=192 | var_explained=0.920 | 5-fold bal_acc=0.6288
  PCA n=256 | var_explained=0.964 | 5-fold bal_acc=0.6095
  PCA n=320 | var_explained=0.991 | 5-fold bal_acc=0.6260

  Best PCA n=128 (CV bal_acc=0.6318)

── Multiple classifiers ──

  Training on DenseNet-TTA feature